In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. Utility Modules (Metrics, Scaling, Splitting)
# ==========================================
def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, and R-squared metrics."""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)

    # Handle edge case where ss_tot is 0 to avoid division by zero
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

class StandardScalerScratch:
    """Standardize features by removing the mean and scaling to unit variance."""
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        # Prevent division by zero for constant features
        self.std_[self.std_ == 0] = 1e-8

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

def chronological_split(X, y, test_ratio=0.2):
    """Split data sequentially to prevent data leakage in time-series forecasting."""
    idx = int(len(X) * (1 - test_ratio))
    return X[:idx], X[idx:], y[:idx], y[idx:]

# ==========================================
# 2. Model Definitions (Ridge & Bayesian)
# ==========================================
class RidgeRegressionScratch:
    """Ridge Regression implemented from scratch using Gradient Descent."""
    def __init__(self, learning_rate=0.005, iterations=2000, l2_penalty=10.0):
        self.lr = learning_rate
        self.iterations = iterations
        self.l2_penalty = l2_penalty

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)

        # Initialize bias to the mean of y to prevent gradient explosion
        self.bias = np.mean(y)

        for _ in range(self.iterations):
            y_pred = np.dot(X, self.weights) + self.bias

            # Compute gradients with L2 regularization
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y)) + (self.l2_penalty / n_samples) * self.weights
            db = (1 / n_samples) * np.sum(y_pred - y)

            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

class BayesianRegressionScratch:
    """Bayesian Linear Regression implemented via exact analytical inference."""
    def __init__(self, alpha=1.0, beta=None):
        self.alpha = alpha
        self.beta = beta
        self.w_mean = None
        self.w_cov = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Automatically infer noise precision (beta) based on target variance
        if self.beta is None:
            self.beta = 1.0 / (np.var(y) + 1e-8)

        # Add a column of ones for the bias (intercept) term
        X_with_bias = np.c_[np.ones(n_samples), X]
        n_features += 1

        I = np.eye(n_features)
        I[0, 0] = 0 # Do not penalize the bias term

        # Compute posterior covariance matrix (S_N) and mean (m_N)
        S_N_inv = self.alpha * I + self.beta * np.dot(X_with_bias.T, X_with_bias)

        # Use pseudo-inverse (pinv) to prevent errors from perfect multicollinearity
        self.w_cov = np.linalg.pinv(S_N_inv)
        self.w_mean = self.beta * np.dot(self.w_cov, np.dot(X_with_bias.T, y))

    def predict(self, X, return_std=False):
        n_samples = X.shape[0]
        X_with_bias = np.c_[np.ones(n_samples), X]
        y_pred_mean = np.dot(X_with_bias, self.w_mean)

        # Calculate predictive uncertainty (standard deviation)
        if return_std:
            y_pred_var = (1 / self.beta) + np.sum(np.dot(X_with_bias, self.w_cov) * X_with_bias, axis=1)
            y_pred_std = np.sqrt(np.abs(y_pred_var))
            return y_pred_mean, y_pred_std

        return y_pred_mean

# ==========================================
# 3. Execution Pipeline & Grid Search
# ==========================================
def train_evaluate_models(df, target_col='Traffic_Volume', date_col='Date', test_ratio=0.2,
                          ridge_params=None, bayes_params=None, verbose=True):
    """
    End-to-end data pipeline. Pass custom parameters via ridge_params and bayes_params.
    """
    # 1. Default parameters if none provided
    if ridge_params is None:
        ridge_params = {'learning_rate': 0.005, 'iterations': 2000, 'l2_penalty': 10.0}
    if bayes_params is None:
        bayes_params = {'alpha': 1.0}

    # 2. Prevent data leakage by ensuring chronological order
    if date_col in df.columns:
        df = df.sort_values(date_col).drop(columns=[date_col])

    # 3. Target extraction and One-Hot Encoding
    y = df[target_col].values
    X_df = pd.get_dummies(df.drop(columns=[target_col]), drop_first=True)
    X = X_df.values.astype(float) # Ensure boolean types from get_dummies are cast to float

    # 4. Chronological split and scaling
    X_train, X_test, y_train, y_test = chronological_split(X, y, test_ratio=test_ratio)
    scaler = StandardScalerScratch()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 5. Train and evaluate Ridge Model
    ridge_model = RidgeRegressionScratch(**ridge_params)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_metrics = calculate_metrics(y_test, ridge_model.predict(X_test_scaled))

    # 6. Train and evaluate Bayesian Model
    bayes_model = BayesianRegressionScratch(**bayes_params)
    bayes_model.fit(X_train_scaled, y_train)
    bayes_preds, bayes_std = bayes_model.predict(X_test_scaled, return_std=True)
    bayes_metrics = calculate_metrics(y_test, bayes_preds)

    # 7. Print Output
    if verbose:
        print(f"--- Stage 1 Evaluation Results for '{target_col}' ---")
        print(f"[Ridge Regression] Parameters: {ridge_params}")
        print(f"    -> MAE: {ridge_metrics['MAE']:.2f} | RMSE: {ridge_metrics['RMSE']:.2f} | R2: {ridge_metrics['R2']:.4f}\n")

        print(f"[Bayesian Linear] Parameters: {bayes_params}")
        print(f"    -> MAE: {bayes_metrics['MAE']:.2f} | RMSE: {bayes_metrics['RMSE']:.2f} | R2: {bayes_metrics['R2']:.4f}\n")

    return {
        "metrics": {"ridge": ridge_metrics, "bayesian": bayes_metrics},
        "models": {"ridge": ridge_model, "bayesian": bayes_model}
    }

# ==========================================
# 4. Example Usage & Parameter Tuning
# ==========================================
if __name__ == "__main__":
    # Example: Creating dummy DataFrame to test the pipeline
    np.random.seed(42)
    dummy_df = pd.DataFrame({
        'Date': pd.date_range(start='2020-01-01', periods=1000, freq='D'),
        'Temp': np.random.normal(60, 15, 1000),
        'Precip': np.random.exponential(0.1, 1000),
        'Is_Peak': np.random.randint(0, 2, 1000),
        'Zipcode': np.random.choice(['10001', '10002', '11201'], 1000),
        'Traffic_Volume': np.random.normal(15000, 3000, 1000)
    })

    # 1. Basic Execution
    print("=== Standard Run ===")
    results = train_evaluate_models(dummy_df, target_col='Traffic_Volume', date_col='Date')

    # 2. Hyperparameter Grid Search Example
    print("=== Custom Grid Search Example (Ridge) ===")
    learning_rates = [0.01, 0.005]
    l2_penalties = [1.0, 10.0]

    best_rmse = float('inf')
    best_params = {}

    for lr in learning_rates:
        for l2 in l2_penalties:
            current_params = {'learning_rate': lr, 'iterations': 1000, 'l2_penalty': l2}

            # Run pipeline silently
            res = train_evaluate_models(dummy_df, target_col='Traffic_Volume', date_col='Date',
                                        ridge_params=current_params, verbose=False)

            rmse = res['metrics']['ridge']['RMSE']
            print(f"Testing LR: {lr}, L2: {l2} => RMSE: {rmse:.2f}")

            if rmse < best_rmse:
                best_rmse = rmse
                best_params = current_params

    print(f"\n>> Best Ridge Parameters Found: {best_params} with RMSE: {best_rmse:.2f}")